In [1]:
pip install -U bertopic sentence-transformers umap-learn hdbscan pandas scikit-learn arabic-reshaper python-bidi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 122.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 133.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
  

In [8]:
import pandas as pd
import re
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_json("all_final_appended.json")

# choose your text column
texts = df["caption_text"].fillna("").astype(str).tolist()

In [9]:
import os
print(os.getcwd())

/content


In [13]:
arabic_diacritics = re.compile(r"[\u0617-\u061A\u064B-\u0652]")

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", " ", text)  # keep hashtag word, remove only #
    text = re.sub(arabic_diacritics, "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

texts = [clean_text(t) for t in texts if len(clean_text(t)) > 5]

In [14]:
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [61]:
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    min_topic_size=8,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(texts)

2026-05-12 17:15:49,195 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/22 [00:00<?, ?it/s]

2026-05-12 17:16:21,309 - BERTopic - Embedding - Completed ✓
2026-05-12 17:16:21,390 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-12 17:16:23,594 - BERTopic - Dimensionality - Completed ✓
2026-05-12 17:16:23,595 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-12 17:16:23,624 - BERTopic - Cluster - Completed ✓
2026-05-12 17:16:23,626 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-12 17:16:23,651 - BERTopic - Representation - Completed ✓


In [62]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,648,0_من_في_كل_متوفر,"[من, في, كل, متوفر, على, مع, الله, مطعم, جنين,...",[شفتي أرقى من هيك موديل قبل؟ 🔥🔥🔥🔥 بضاعة العيد ...
1,1,14,1_عبر رسائل_عبر_رسائل الصفحة_رسائل,"[عبر رسائل, عبر, رسائل الصفحة, رسائل, الصفحة, ...",[شنط مميزة شغل يدوي مميز 🤎 جنين برج الساعة الط...
2,2,14,2_الطابق_الثاني_آخر_متوفر لجميع,"[الطابق, الثاني, آخر, متوفر لجميع, متوفرة داخل...",[أكبر وأوسع تشكيلة من العبايات الخليجية الفخمة...
3,3,9,3_البيرة_022423366_0569690061_dco بجانب,"[البيرة, 022423366, 0569690061, dco بجانب, 059...",[جمعة مباركة البيرة - الطلوع قرب مدخل DCO | بج...


In [63]:
topic_model.get_topic(0)

[('من', np.float64(0.14264143397979037)),
 ('في', np.float64(0.1294714709926757)),
 ('كل', np.float64(0.0870124822050975)),
 ('متوفر', np.float64(0.08696603297382209)),
 ('على', np.float64(0.08366086968828877)),
 ('مع', np.float64(0.08148993371087289)),
 ('الله', np.float64(0.07595433341924883)),
 ('مطعم', np.float64(0.0701068075823093)),
 ('جنين', np.float64(0.06598615110021028)),
 ('عرض', np.float64(0.06588649653400516))]

In [64]:
topic_model.visualize_topics()

In [65]:
topic_model.visualize_barchart(top_n_topics=10)

In [39]:
df_clean = df[df["caption_text"].fillna("").astype(str).str.len() > 5].copy()
df_clean["topic_id"] = topics

topic_info = topic_model.get_topic_info()
df_clean = df_clean.merge(
    topic_info[["Topic", "Name"]],
    left_on="topic_id",
    right_on="Topic",
    how="left"
)

df_clean.to_csv("dataset_with_topics.csv", index=False)

In [40]:
topic_info = topic_model.get_topic_info()

for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue

    keywords = topic_model.get_topic(topic_id)
    docs = topic_model.get_representative_docs(topic_id)

    print("\nTopic:", topic_id)
    print("Keywords:", keywords[:10])
    print("Examples:", docs[:3])


Topic: 0
Keywords: [('the', np.float64(0.06641563255265041)), ('gaza', np.float64(0.06034477058792767)), ('of', np.float64(0.0578593460122206)), ('in', np.float64(0.05681257820162266)), ('to', np.float64(0.05359140098930889)), ('0566000650', np.float64(0.04284935737091275)), ('0599420500', np.float64(0.04284935737091275)), ('الشرقي', np.float64(0.039994803967967)), ('and', np.float64(0.03884851969232496)), ('الثورة', np.float64(0.038564421633821476))]
Examples: ['المذاق الفاخر يتميز به في جميع الأصناف الشرقية و الغربية في مزاج الشرقي. للحجز والاستفسار للتواصل: 0599420500، 0566000650، 082825003 العنوان: غزة، تقاطع شارع النصر مع شارع الثورة - بناية بدري وهبة.', 'استمتع بتجربة متميزة مع مزاج الشرقي ونكهاته الشهية والأصناف الشرقية الأصيلة والمتنوعة. للحجز والاستفسار للتواصل: 0599420500، 0566000650، 082825003 العنوان: غزة، تقاطع شارع النصر مع شارع الثورة - بناية بدري وهبة.', 'استمتعوا بلذة الأصناف الغربية من مزاج الشرقي. للحجز والاستفسار التواصل: 0599420500، 0566000650، 082825003 العنوان: 